# FinClub Open Project 2026 — Final Private-Safe IV Surface Imputation

This notebook generates the final selected Kaggle submission:

`submission.csv`

## Problem summary

The task is to predict missing implied volatility values in a Nifty options dataset. The data is naturally structured as an implied volatility surface across:

- timestamps,
- strikes,
- option type: CE / PE.

Instead of treating missing IV values as unrelated table cells, this notebook reconstructs the IV surface using financial structure.

## Core financial intuition

Implied volatility usually varies smoothly across nearby strikes at the same timestamp and often shows smile/skew-like curvature. Therefore, missing IV values can be estimated using:

- same-timestamp nearby strikes,
- local strike-wise interpolation,
- polynomial smile/skew fitting,
- past-only time persistence.

## Overfitting control

The public leaderboard only measures part of the test data. To reduce public leaderboard overfitting, this notebook uses internal masked validation:

1. Random masking,
2. Actual missing-pattern masking,
3. Edge-strike masking.

The final model is selected using this internal validation, not by blindly tuning on public leaderboard score.

## Cell 1 — Import libraries and define file paths

This cell imports all required Python libraries and defines the input/output filenames.

**What is being used here:**
- `pandas` and `numpy` for data handling and numerical operations.
- `re` for parsing option column names such as strike and option type.
- `Akima1DInterpolator` for smooth local strike-wise interpolation.
- `minimize` from SciPy to learn ensemble weights using validation loss.
- `mean_squared_error` and `mean_absolute_error` for internal validation.

**Important output files:**
- `submission.csv`: final Kaggle submission file.
- `filled_dataset.csv`: dataset with missing IV values filled.
- `v3_validation_report.csv`: internal validation results.
- `v3_ensemble_weights.csv`: learned ensemble weights.

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from scipy.interpolate import Akima1DInterpolator
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error

DATA_PATH = "dataset.csv"
SAMPLE_SUB_PATH = "sandbox_solution.csv"

OUT_PRIVATE_SAFE = "submission.csv"
OUT_FILLED = "filled_dataset.csv"
OUT_REPORT = "v3_validation_report.csv"
OUT_WEIGHTS = "v3_ensemble_weights.csv"

## Cell 2 — Load data and extract option metadata

This cell loads the dataset and sample submission, then prepares the option matrix.

**Main steps:**
1. Reads `dataset.csv` and `sandbox_solution.csv`.
2. Parses the timestamp using `dayfirst=True`, because the dates are in Indian-style format.
3. Keeps the original `datetime` string unchanged because the Kaggle submission IDs are built from it.
4. Extracts strike, expiry, and option type from each option column.
5. Separates option columns into `CE` and `PE` groups.
6. Converts the option IV values into a matrix `Y0`, where rows are timestamps and columns are option contracts.

**Financial intuition:**
Calls and puts are handled separately because they may have different strike ranges and missing-value patterns. The model reconstructs the IV surface across strikes within each option type.

In [ ]:
df = pd.read_csv(DATA_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# Parsing, but keeping original datetime string for submission IDs.
df["datetime_parsed"] = pd.to_datetime(df["datetime"], dayfirst=True)
df = df.sort_values("datetime_parsed").reset_index(drop=True)

option_cols = [c for c in df.columns if c not in ["datetime", "underlying_price", "datetime_parsed"]]

def parse_option(col):
    m = re.match(r"NIFTY(\d{2}[A-Z]{3}\d{2})(\d+)(CE|PE)$", col)
    if m is None:
        raise ValueError(f"Could not parse option column: {col}")
    return m.group(1), int(m.group(2)), m.group(3)

meta = pd.DataFrame([
    {"column": c, "expiry": parse_option(c)[0], "strike": parse_option(c)[1], "type": parse_option(c)[2]}
    for c in option_cols
])

col_index = {c: i for i, c in enumerate(option_cols)}
groups = {}
for opt_type in ["CE", "PE"]:
    tmp = meta.loc[meta["type"].eq(opt_type)].sort_values("strike")
    groups[opt_type] = {
        "columns": tmp["column"].tolist(),
        "idx": np.array([col_index[c] for c in tmp["column"]]),
        "strike": tmp["strike"].to_numpy(dtype=float),
    }

Y0 = df[option_cols].to_numpy(dtype=float)
expiry_dt = pd.Timestamp("2026-01-27 15:30")
days_to_expiry = ((expiry_dt - df["datetime_parsed"]).dt.total_seconds() / 86400).to_numpy()

print("Rows:", len(df))
print("Option columns:", len(option_cols))
print("Missing cells to predict:", int(np.isnan(Y0).sum()))

Rows: 975
Option columns: 28
Missing cells to predict: 5460


## Cell 3 — Define base interpolation and smoothing methods

This cell defines the core imputation functions used to fill missing IV values.

**Functions in this cell:**
- `fit_1d()`: fits a one-dimensional curve across strikes for a single timestamp.
- `row_fill_matrix()`: applies strike-wise interpolation separately for CE and PE at every timestamp.
- `past_time_fill_matrix()`: fills missing values using only previous timestamps for the same option contract.

**Methods used:**
- Linear interpolation: simple and stable.
- Akima interpolation: smooth local interpolation that avoids unnecessary oscillations.
- Polynomial fitting: captures smile/skew-like curvature.
- Past-time fill: uses short-term persistence of implied volatility.

**Lookahead safety:**
The time-based fill uses forward fill from the past only. It does not use future timestamps to predict earlier missing values.

In [ ]:
def fit_1d(x_obs, y_obs, x_all, method="linear", degree=2):
    x_obs = np.asarray(x_obs, dtype=float)
    y_obs = np.asarray(y_obs, dtype=float)
    x_all = np.asarray(x_all, dtype=float)

    if len(y_obs) == 0:
        return np.full_like(x_all, np.nan, dtype=float)
    if len(y_obs) == 1:
        return np.full_like(x_all, y_obs[0], dtype=float)

    order = np.argsort(x_obs)
    x_obs = x_obs[order]
    y_obs = y_obs[order]

    try:
        if method == "linear":
            pred = np.interp(x_all, x_obs, y_obs)

            left = x_all < x_obs[0]
            right = x_all > x_obs[-1]

            if left.any():
                slope = (y_obs[1] - y_obs[0]) / (x_obs[1] - x_obs[0])
                pred[left] = y_obs[0] + slope * (x_all[left] - x_obs[0])

            if right.any():
                slope = (y_obs[-1] - y_obs[-2]) / (x_obs[-1] - x_obs[-2])
                pred[right] = y_obs[-1] + slope * (x_all[right] - x_obs[-1])

        elif method == "akima":
            pred = Akima1DInterpolator(x_obs, y_obs)(x_all)
            if np.isnan(pred).any():
                fallback = fit_1d(x_obs, y_obs, x_all, method="linear")
                pred = np.where(np.isnan(pred), fallback, pred)

        elif method == "poly":
            d = min(degree, len(y_obs) - 1)
            mu = x_obs.mean()
            sd = x_obs.std() if x_obs.std() > 0 else 1.0
            coef = np.polyfit((x_obs - mu) / sd, y_obs, d)
            pred = np.polyval(coef, (x_all - mu) / sd)

        else:
            raise ValueError(method)

    except Exception:
        pred = fit_1d(x_obs, y_obs, x_all, method="linear")

    return np.clip(pred, 0.005, 8.0)

def row_fill_matrix(Y, method="linear", degree=2):
    out = Y.copy()
    for opt_type, g in groups.items():
        idx = g["idx"]
        x = g["strike"]
        for i in range(Y.shape[0]):
            y = Y[i, idx]
            observed = ~np.isnan(y)
            if observed.all():
                continue
            pred = fit_1d(x[observed], y[observed], x, method=method, degree=degree)
            out[i, idx[~observed]] = pred[~observed]
    return np.clip(out, 0.005, 8.0)

def past_time_fill_matrix(Y, fallback_matrix=None):
    # No future-time interpolation. Used ffill only.
    # Leading values without past observations fall back to same-timestamp row-surface fill.
    if fallback_matrix is None:
        fallback_matrix = row_fill_matrix(Y, method="linear")

    out = Y.copy()
    for j in range(Y.shape[1]):
        s = pd.Series(out[:, j]).ffill()
        vals = s.to_numpy(dtype=float)
        still_missing = np.isnan(vals)
        if still_missing.any():
            vals[still_missing] = fallback_matrix[still_missing, j]
        out[:, j] = vals
    return np.clip(out, 0.005, 8.0)

## Cell 4 — Define the V2-style strict surface model

This cell defines `v2_strict_matrix()`, a rule-based surface imputation model.

**Logic used:**
- For interior missing strikes, the model uses mostly Akima interpolation because nearby strikes exist on both sides.
- For edge missing strikes, the model uses a safer blend of linear extrapolation, polynomial smile fit, and past-time fill.

**Why edge strikes are treated differently:**
Edge strikes involve extrapolation, not interpolation. They are usually riskier because there is no observed strike on one side. So the model uses more conservative smoothing there.

In [4]:
def v2_strict_matrix(Y):
    linear = row_fill_matrix(Y, method="linear")
    akima = row_fill_matrix(Y, method="akima")
    poly2 = row_fill_matrix(Y, method="poly", degree=2)
    past = past_time_fill_matrix(Y, fallback_matrix=linear)

    out = Y.copy()

    for i in range(Y.shape[0]):
        for opt_type, g in groups.items():
            idx = g["idx"]
            y = Y[i, idx]
            observed = ~np.isnan(y)

            for local_pos, j in enumerate(idx):
                if not np.isnan(Y[i, j]):
                    continue

                is_interior = observed[:local_pos].any() and observed[local_pos + 1:].any()

                if is_interior:
                    val = 0.8 * akima[i, j] + 0.2 * linear[i, j]
                else:
                    val = 0.8 * linear[i, j] + 0.1 * poly2[i, j] + 0.1 * past[i, j]

                out[i, j] = val

    return np.clip(out, 0.005, 8.0)

## Cell 5 — Create validation masks and define grouping logic

This cell builds the internal validation system.

**Validation idea:**
Since the true missing IV values are hidden, we simulate the task by hiding some known IV values and checking how accurately the model can recover them.

**Three validation styles:**
1. `random`: randomly hides known IV cells.
2. `pattern`: hides values in a way that mimics the actual missing pattern.
3. `edge`: stresses edge strikes, where extrapolation is most difficult.

**Grouping logic:**
The function `group_label_for_cell()` categorizes each missing cell based on:
- whether it is an edge or interior strike,
- whether it is near expiry,
- whether the row has low observed data.

This allows different ensemble weights for different market situations.

In [5]:
def make_holdout_masks(Y, n_folds=2, mode="pattern", frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    observed = ~np.isnan(Y)
    masks = []

    for _ in range(n_folds):
        holdout = np.zeros_like(observed, dtype=bool)

        if mode == "random":
            positions = np.argwhere(observed)
            k = int(len(positions) * frac)
            chosen = rng.choice(len(positions), size=k, replace=False)
            holdout[positions[chosen, 0], positions[chosen, 1]] = True

        elif mode == "pattern":
            for i in range(Y.shape[0]):
                for opt_type, g in groups.items():
                    idx = g["idx"]
                    available = idx[observed[i, idx]]
                    actual_missing_count = np.isnan(Y[i, idx]).sum()
                    mask_count = min(int(actual_missing_count), max(0, len(available) // 3))
                    if mask_count > 0 and len(available) > 2:
                        chosen = rng.choice(available, size=mask_count, replace=False)
                        holdout[i, chosen] = True

        elif mode == "edge":
            for i in range(Y.shape[0]):
                for opt_type, g in groups.items():
                    idx = g["idx"]
                    available_local_pos = np.where(observed[i, idx])[0]
                    if len(available_local_pos) <= 3:
                        continue
                    edge_positions = [p for p in available_local_pos if p in [0, 1, len(idx)-2, len(idx)-1]]
                    if edge_positions:
                        holdout[i, idx[rng.choice(edge_positions)]] = True
                    if rng.random() < 0.60:
                        interior_positions = [p for p in available_local_pos if p not in edge_positions]
                        if interior_positions:
                            holdout[i, idx[rng.choice(interior_positions)]] = True

        masks.append(holdout)

    return masks

def simplex_fit(P, y):
    k = P.shape[1]
    def objective(w):
        return np.mean((P @ w - y) ** 2)
    result = minimize(
        objective,
        np.ones(k) / k,
        method="SLSQP",
        bounds=[(0, 1)] * k,
        constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return result.x

def is_edge_position(Y, i, j):
    opt_type = meta.iloc[j]["type"]
    idx = groups[opt_type]["idx"]
    local_pos = int(np.where(idx == j)[0][0])
    observed = ~np.isnan(Y[i, idx])
    return not (observed[:local_pos].any() and observed[local_pos + 1:].any())

def group_label_for_cell(Y, i, j):
    edge = is_edge_position(Y, i, j)
    opt_type = meta.iloc[j]["type"]
    idx = groups[opt_type]["idx"]
    n_observed_same_type = np.sum(~np.isnan(Y[i, idx]))
    near_expiry = days_to_expiry[i] <= 1.5
    low_observation = n_observed_same_type <= 8
    return int(edge) * 4 + int(near_expiry) * 2 + int(low_observation)

## Cell 6 — Run internal validation and collect candidate predictions

This cell evaluates all candidate methods on the masked validation tasks.

**Candidate methods tested:**
- `linear`
- `akima`
- `poly2`
- `poly3`
- `time_past`
- `v2_strict`

For each validation mask, the notebook:
1. hides known IV values,
2. predicts them using each candidate method,
3. compares predictions with the actual known IV values,
4. stores MSE and MAE in `validation_report`.

This cell is important because the final method is selected using internal validation, not just public leaderboard score.

In [6]:
def collect_validation_predictions(Y):
    candidate_names = ["linear", "akima", "poly2", "poly3", "time_past"]
    masks, modes = [], []

    for mode, seed in [("pattern", 101), ("random", 102), ("edge", 103)]:
        new_masks = make_holdout_masks(Y, n_folds=2, mode=mode, frac=0.15, seed=seed)
        masks.extend(new_masks)
        modes.extend([mode] * len(new_masks))

    records, prediction_rows, y_rows, group_rows = [], [], [], []

    for mask_id, (holdout, mode) in enumerate(zip(masks, modes)):
        Y_train = Y.copy()
        Y_train[holdout] = np.nan

        linear = row_fill_matrix(Y_train, method="linear")
        candidates = {
            "linear": linear,
            "akima": row_fill_matrix(Y_train, method="akima"),
            "poly2": row_fill_matrix(Y_train, method="poly", degree=2),
            "poly3": row_fill_matrix(Y_train, method="poly", degree=3),
            "time_past": past_time_fill_matrix(Y_train, fallback_matrix=linear),
            "v2_strict": v2_strict_matrix(Y_train),
        }

        for name, pred in candidates.items():
            records.append({
                "validation_mode": mode,
                "fold": mask_id,
                "method": name,
                "mse": mean_squared_error(Y[holdout], pred[holdout]),
                "mae": mean_absolute_error(Y[holdout], pred[holdout]),
                "n_holdout": int(holdout.sum()),
            })

        for i, j in np.argwhere(holdout):
            prediction_rows.append([candidates[name][i, j] for name in candidate_names])
            y_rows.append(Y[i, j])
            group_rows.append(group_label_for_cell(Y_train, i, j))

    return candidate_names, pd.DataFrame(records), np.asarray(prediction_rows), np.asarray(y_rows), np.asarray(group_rows)

candidate_names, validation_report, P, y_valid, group_labels = collect_validation_predictions(Y0)
validation_report.groupby(["validation_mode", "method"])[["mse", "mae"]].mean()

mse       mae
validation_mode method                       
edge            akima      0.000088  0.002763
                linear     0.000083  0.002869
                poly2      0.000161  0.004151
                poly3      0.000138  0.003237
                time_past  0.004089  0.009572
                v2_strict  0.000119  0.002837
pattern         akima      0.000087  0.002276
                linear     0.000086  0.002564
                poly2      0.000119  0.003436
                poly3      0.000110  0.002733
                time_past  0.002596  0.008860
                v2_strict  0.000090  0.002294
random          akima      0.000088  0.002202
                linear     0.000089  0.002457
                poly2      0.000110  0.003407
                poly3      0.000075  0.002384
                time_past  0.001968  0.008671
                v2_strict  0.000073  0.002170

## Cell 7 — Learn validation-based ensemble weights

This cell learns the final ensemble weights.

**What happens here:**
- Predictions from multiple base methods are combined.
- The weights are learned by minimizing validation MSE.
- The weights are constrained to be non-negative and sum to 1.

**Why this is robust:**
Instead of manually guessing how much weight to give each method, the notebook learns weights from masked validation. Different groups of missing cells receive different weights, which makes the final model adaptive.

In [7]:
simple_group_labels = (group_labels >= 4).astype(int)

weights_simple = {}
for label in sorted(np.unique(simple_group_labels)):
    mask = simple_group_labels == label
    weights_simple[int(label)] = simplex_fit(P[mask], y_valid[mask])

weights_private_safe = {}
for label in sorted(np.unique(group_labels)):
    mask = group_labels == label
    weights_private_safe[int(label)] = simplex_fit(P[mask], y_valid[mask])

def make_weight_table(weights_dict, model_name):
    rows = []
    for label, weights in weights_dict.items():
        row = {"model": model_name, "group_label": int(label)}
        for name, w in zip(candidate_names, weights):
            row[name] = float(w)
        rows.append(row)
    return pd.DataFrame(rows)

weights_df = pd.concat([
    make_weight_table(weights_simple, "simple_edge_interior"),
    make_weight_table(weights_private_safe, "private_safe_adaptive"),
], ignore_index=True)

weights_df

,model,group_label,linear,akima,poly2,poly3,time_past
0,simple_edge_interior,0,1.298055e-01,1.826963e-01,1.727835e-01,0.499956,0.014758
1,simple_edge_interior,1,2.200858e-01,2.200859e-01,2.272677e-01,0.312386,0.020175
2,private_safe_adaptive,0,2.378862e-01,2.402935e-01,2.349348e-01,0.241374,0.045512
3,private_safe_adaptive,1,2.301049e-01,2.453450e-01,2.287995e-01,0.243251,0.052500
4,private_safe_adaptive,2,1.038279e-01,2.644347e-01,1.214306e-17,0.614096,0.017641
5,private_safe_adaptive,3,8.084280e-18,1.063750e-01,3.936978e-01,0.486986,0.012941
6,private_safe_adaptive,4,1.100655e-17,3.458858e-18,3.581467e-01,0.483955,0.157898
7,private_safe_adaptive,5,1.075964e-01,1.075964e-01,4.118955e-01,0.135695,0.237217
8,private_safe_adaptive,6,3.592632e-01,3.592632e-01,0.000000e+00,0.280363,0.001110
9,private_safe_adaptive,7,1.506364e-01,1.506364e-01,3.530434e-01,0.302919,0.042765


## Cell 8 — Generate final filled IV matrix

This cell applies the learned ensemble weights to the actual missing values in the dataset.

**Main steps:**
1. Computes full-data predictions from each base method.
2. Finds all missing cells in the original IV matrix.
3. For each missing cell, identifies its adaptive group label.
4. Applies the corresponding validation-learned ensemble weights.
5. Stores the final filled values in `Y_private_safe`.

**Final model equation:**

`Final IV = w1 × linear + w2 × akima + w3 × poly2 + w4 × poly3 + w5 × time_past`

The weights depend on the type of missing cell.

**Important:** This cleaned final notebook creates only one submission file: `submission.csv`.


In [8]:
linear_full = row_fill_matrix(Y0, method="linear")
base_fills = {
    "linear": linear_full,
    "akima": row_fill_matrix(Y0, method="akima"),
    "poly2": row_fill_matrix(Y0, method="poly", degree=2),
    "poly3": row_fill_matrix(Y0, method="poly", degree=3),
    "time_past": past_time_fill_matrix(Y0, fallback_matrix=linear_full),
}

Y_private_safe = Y0.copy()

missing_positions = np.argwhere(np.isnan(Y0))

for i, j in missing_positions:
    preds = np.asarray([base_fills[name][i, j] for name in candidate_names])
    adaptive_label = group_label_for_cell(Y0, i, j)

    Y_private_safe[i, j] = np.clip(preds @ weights_private_safe[adaptive_label], 0.005, 8.0)

## Cell 9 — Create the final submission CSV and save validation artifacts

This cell converts the filled IV matrix into the exact Kaggle submission format.

**Important details:**
- Submission IDs are created using the original datetime string and option column name.
- The notebook checks that there are no missing predictions in the final submission.
- Only one final Kaggle submission file is created: `submission.csv`.

**Additional saved files:**
- `filled_dataset.csv`: useful for inspection.
- `v3_validation_report.csv`: useful for explaining validation.
- `v3_ensemble_weights.csv`: useful for explaining the ensemble.

In [9]:
def matrix_to_submission(Y, output_path):
    pred_map = {}
    for i, j in missing_positions:
        pred_map[f"{df.loc[i, 'datetime']}||{option_cols[j]}"] = float(Y[i, j])
    sub = sample_sub[["id"]].copy()
    sub["value"] = sub["id"].map(pred_map)
    assert sub["value"].isna().sum() == 0
    sub.to_csv(output_path, index=False)
    return sub

sub_private = matrix_to_submission(Y_private_safe, OUT_PRIVATE_SAFE)

filled_df = df.drop(columns=["datetime_parsed"]).copy()
filled_values = filled_df[option_cols].to_numpy(dtype=float)
filled_values[np.isnan(Y0)] = Y_private_safe[np.isnan(Y0)]
filled_df[option_cols] = filled_values
filled_df.to_csv(OUT_FILLED, index=False)

validation_report.to_csv(OUT_REPORT, index=False)
weights_df.to_csv(OUT_WEIGHTS, index=False)

print("Saved:", OUT_PRIVATE_SAFE)
print("NaNs:", sub_private["value"].isna().sum())
sub_private["value"].describe()

Saved: submission.csv
NaNs: 0


count    5460.000000
mean        0.187183
std         0.256982
min         0.074394
25%         0.110617
50%         0.130980
75%         0.167785
max         5.787525
Name: value, dtype: float64

## Final submission file

`submission.csv`

for Kaggle submission.

The files `v3_validation_report.csv` and `v3_ensemble_weights.csv` are not submission files. They are kept only to explain the model, validation strategy, and learned ensemble weights in the final report.

The final notebook is intentionally structured cell-by-cell so that each modelling step is clear and reproducible.

**Filename note:** The final submission file is intentionally named `submission.csv`, and the filled dataset file is named `filled_dataset.csv`, matching the expected converter/submission naming convention.
